# Large-universe monthly rebalance notebook

This version does three things you asked for:
- includes a pip install cell at the beginning
- treats BIST as a basket of stocks, not a single index ticker
- rebuilds the portfolio each month using only information available up to that month

Important honesty note:
- I cannot verify the latest official BIST 100 constituent list live from here.
- The stock list below is the earlier large BIST universe from the previous notebook, not a guaranteed current official 100-member list.
- TEFAS is left as a manual optional input because provider coverage is inconsistent.


In [ ]:
!pip -q install openbb yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px

from pypfopt import expected_returns, risk_models, EfficientFrontier

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## Universe config

Replace `BIST100_TICKERS` with the current official list if you have it.
Add TEFAS fund symbols only if your data source supports them or if you provide manual TEFAS prices later.


In [ ]:
BIST100_TICKERS = ['AEFES.IS', 'AGHOL.IS', 'AKBNK.IS', 'AKFYE.IS', 'AKSA.IS', 'AKSEN.IS', 'ALARK.IS', 'ARCLK.IS', 'ASELS.IS', 'ASTOR.IS', 'BIMAS.IS', 'BOBET.IS', 'CCOLA.IS', 'CIMSA.IS', 'CWENE.IS', 'DOAS.IS', 'DOHOL.IS', 'ECILC.IS', 'EGEEN.IS', 'EKGYO.IS', 'ENERY.IS', 'ENJSA.IS', 'ENKAI.IS', 'EREGL.IS', 'EUPWR.IS', 'FROTO.IS', 'GARAN.IS', 'GESAN.IS', 'GUBRF.IS', 'HALKB.IS', 'HEKTS.IS', 'ISCTR.IS', 'ISMEN.IS', 'KAYSE.IS', 'KCAER.IS', 'KCHOL.IS', 'KLSER.IS', 'KOZAA.IS', 'KOZAL.IS', 'KRDMD.IS', 'MAVI.IS', 'MGROS.IS', 'MIATK.IS', 'ODAS.IS', 'OTKAR.IS', 'OYAKC.IS', 'PETKM.IS', 'PGSUS.IS', 'REEDR.IS', 'SASA.IS', 'SAHOL.IS', 'SDTTR.IS', 'SISE.IS', 'SKBNK.IS', 'SMRTG.IS', 'SOKM.IS', 'TAVHL.IS', 'TCELL.IS', 'THYAO.IS', 'TKFEN.IS', 'TOASO.IS', 'TSKB.IS', 'TTKOM.IS', 'TTRAK.IS', 'TUPRS.IS', 'ULKER.IS', 'VAKBN.IS', 'VESBE.IS', 'VESTL.IS', 'YKBNK.IS', 'ZOREN.IS']

FX_TICKERS = {'USDTRY': 'USDTRY=X', 'EURTRY': 'EURTRY=X'}
METAL_USD_TICKERS = {'ALTIN_USD': 'GC=F', 'GUMUS_USD': 'SI=F', 'PLATIN_USD': 'PL=F'}
TEFAS_FUND_TICKERS = []
LOOKBACK_DAYS = 252
LOOKBACK_MONTHS = 12
MAX_WEIGHT = 0.10
print('BIST starter universe size:', len(BIST100_TICKERS))


In [ ]:
def fetch_close_series(symbol, period='10y'):
    df = yf.download(symbol, period=period, auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs('Close', axis=1, level=0) if 'Close' in level0 else df.iloc[:, :1]
    else:
        close = df[['Close']] if 'Close' in df.columns else df.iloc[:, :1]
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]
    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def safe_asset_name(symbol):
    return symbol.replace('.IS', '').replace('=X', '').replace('=F', '')

rows = []
series_list = []
for sym in BIST100_TICKERS:
    s = fetch_close_series(sym)
    rows.append({'source': sym, 'asset': safe_asset_name(sym), 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty', 'group': 'BIST'})
    if s is not None and len(s) >= 180:
        s = s.copy(); s.name = safe_asset_name(sym); series_list.append(s)
fx_series = {}
for label, sym in FX_TICKERS.items():
    s = fetch_close_series(sym)
    fx_series[label] = s
    rows.append({'source': sym, 'asset': label, 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty', 'group': 'FX'})
    if s is not None and len(s) >= 180:
        s = s.copy(); s.name = label; series_list.append(s)
metal_usd = {}
for label, sym in METAL_USD_TICKERS.items():
    s = fetch_close_series(sym)
    metal_usd[label] = s
usdtry = fx_series.get('USDTRY')
for try_label, usd_label in [('ALTIN_TRY', 'ALTIN_USD'), ('GUMUS_TRY', 'GUMUS_USD'), ('PLATIN_TRY', 'PLATIN_USD')]:
    base = metal_usd.get(usd_label)
    if base is not None and usdtry is not None:
        s = (base * usdtry).dropna()
        s.name = try_label
        rows.append({'source': f'{usd_label} * USDTRY', 'asset': try_label, 'n': len(s), 'start': str(s.index.min().date()), 'end': str(s.index.max().date()), 'status': 'ok', 'group': 'METAL_TRY'})
        if len(s) >= 180:
            series_list.append(s)
coverage = pd.DataFrame(rows)
display(coverage)
prices = pd.concat(series_list, axis=1).sort_index().ffill()
common_start = prices.apply(lambda s: s.first_valid_index()).max()
prices = prices.loc[common_start:].dropna()
returns = prices.pct_change().dropna()
print('Eligible asset count:', prices.shape[1])
display(prices.tail())


In [ ]:
def build_portfolio(train_returns):
    train_returns = train_returns.dropna(axis=1, how='all').dropna(how='any')
    if train_returns.shape[1] < 2:
        raise ValueError('Need at least 2 valid assets in the training window.')
    prices_window = (1 + train_returns).cumprod()
    mu = expected_returns.mean_historical_return(prices_window, frequency=252)
    S = risk_models.CovarianceShrinkage(prices_window).ledoit_wolf()
    ef = EfficientFrontier(mu, S, weight_bounds=(0, MAX_WEIGHT))
    ef.max_sharpe()
    weights = ef.clean_weights()
    weights = {k: float(v) for k, v in weights.items() if v is not None and float(v) > 0}
    chosen_assets = list(weights.keys())
    return chosen_assets, weights

rebal_dates = returns.resample('M').last().index
selected_history = []
weight_history = []
portfolio_rets = []
for i in range(LOOKBACK_MONTHS, len(rebal_dates) - 1):
    rebalance_date = rebal_dates[i]
    next_rebalance_date = rebal_dates[i + 1]
    train = returns.loc[:rebalance_date].tail(LOOKBACK_DAYS)
    valid_cols = train.columns[train.notna().sum() > max(60, LOOKBACK_DAYS // 3)]
    train = train[valid_cols].dropna(how='any')
    if train.empty or train.shape[1] < 2:
        continue
    try:
        chosen_assets, weights = build_portfolio(train)
    except Exception as e:
        print(f'Skipped {rebalance_date.date()} due to optimizer error: {e}')
        continue
    selected_history.append(pd.Series(chosen_assets, name=rebalance_date))
    weight_history.append(pd.Series(weights, name=rebalance_date))
    hold = returns.loc[(returns.index > rebalance_date) & (returns.index <= next_rebalance_date), list(weights.keys())].copy()
    if hold.empty:
        continue
    hold = hold.fillna(0.0)
    aligned_w = pd.Series(weights).reindex(hold.columns).fillna(0.0)
    realized = hold.mul(aligned_w, axis=1).sum(axis=1)
    portfolio_rets.append(realized)
portfolio_rets = pd.concat(portfolio_rets).sort_index()
selected_df = pd.DataFrame(selected_history)
weights_df = pd.DataFrame(weight_history).fillna(0.0)
equity_curve = (1 + portfolio_rets).cumprod()
display(selected_df.tail())
display(weights_df.tail())


In [ ]:
ann_ret = (1 + portfolio_rets).prod() ** (252 / len(portfolio_rets)) - 1
ann_vol = portfolio_rets.std() * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol != 0 else np.nan
max_dd = (equity_curve / equity_curve.cummax() - 1).min()
summary = pd.DataFrame({'metric': ['CAGR', 'Volatility', 'Sharpe', 'Max Drawdown', 'Rebalance Count'], 'value': [ann_ret, ann_vol, sharpe, max_dd, len(weights_df)]})
display(summary)
print('Chosen assets each month:')
display(selected_df)
print('Weights each month:')
display(weights_df.round(4))
fig = px.line(x=equity_curve.index, y=equity_curve.values, labels={'x': 'Date', 'y': 'Growth of 1'}, title='Monthly Rebalanced Equity Curve')
fig.show()
